In [0]:
# ================================================
# NYC Taxi Data Analysis | Databricks + PySpark
# ================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, round, unix_timestamp, when

spark = SparkSession.builder.appName("NYCTaxi").getOrCreate()

# 1. LOAD DATA
df = spark.read.table("samples.nyctaxi.trips")
print(f"Raw records: {df.count():,}")
df.printSchema()

# 2. REMOVE NULLS & DUPLICATES
df = df.dropna()
df = df.dropDuplicates()
print(f"After cleaning: {df.count():,}")

# 3. FILTER OUTLIERS
df = df.filter(
    (col("fare_amount").between(1, 200))    &
    (col("trip_distance").between(0.1, 50))
)
print(f"After filtering: {df.count():,}")

# 4. FEATURE ENGINEERING
df = df \
    .withColumn("trip_duration_mins",
        round((unix_timestamp("tpep_dropoff_datetime") -
               unix_timestamp("tpep_pickup_datetime")) / 60, 2)) \
    .withColumn("fare_per_mile",
        round(col("fare_amount") / (col("trip_distance") + 0.01), 2)) \
    .withColumn("is_rush_hour",
        when(hour("tpep_pickup_datetime").isin(
            list(range(7,10)) + list(range(17,20))), 1).otherwise(0))

df.createOrReplaceTempView("nyctaxi")
print("✅ Features added, view registered")
df.show(5)

# 5. PEAK DEMAND HOURS
print("--- Peak Demand Hours ---")
spark.sql("""
    SELECT
        HOUR(tpep_pickup_datetime)       AS pickup_hour,
        COUNT(*)                         AS total_trips,
        ROUND(AVG(fare_amount), 2)       AS avg_fare
    FROM nyctaxi
    GROUP BY pickup_hour
    ORDER BY total_trips DESC
""").show()

# 6. DAILY REVENUE TREND
print("--- Daily Revenue ---")
spark.sql("""
    SELECT
        DATE(tpep_pickup_datetime)       AS trip_date,
        COUNT(*)                         AS total_trips,
        ROUND(SUM(fare_amount), 2)       AS daily_revenue,
        ROUND(AVG(fare_amount), 2)       AS avg_fare
    FROM nyctaxi
    GROUP BY trip_date
    ORDER BY trip_date
""").show()

# 7. WEEKDAY vs WEEKEND
print("--- Weekday vs Weekend ---")
spark.sql("""
    SELECT
        CASE WHEN DAYOFWEEK(tpep_pickup_datetime) IN (1,7)
             THEN 'Weekend' ELSE 'Weekday'
        END                              AS day_type,
        COUNT(*)                         AS total_trips,
        ROUND(AVG(fare_amount), 2)       AS avg_fare,
        ROUND(AVG(trip_duration_mins), 2) AS avg_duration_mins
    FROM nyctaxi
    GROUP BY day_type
""").show()

# 8. FARE SEGMENTATION
print("--- Fare Tiers ---")
spark.sql("""
    SELECT
        CASE
            WHEN fare_amount < 10 THEN 'Budget  (< $10)'
            WHEN fare_amount < 25 THEN 'Mid     ($10-$25)'
            ELSE                       'Premium ($25+)'
        END                            AS fare_tier,
        COUNT(*)                       AS total_trips,
        ROUND(AVG(trip_distance), 2)   AS avg_distance
    FROM nyctaxi
    GROUP BY fare_tier
    ORDER BY total_trips DESC
""").show()

Raw records: 21,932
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_zip: integer (nullable = true)
 |-- dropoff_zip: integer (nullable = true)

After cleaning: 21,932
After filtering: 21,824
✅ Features added, view registered
+--------------------+---------------------+-------------+-----------+----------+-----------+------------------+-------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|trip_duration_mins|fare_per_mile|is_rush_hour|
+--------------------+---------------------+-------------+-----------+----------+-----------+------------------+-------------+------------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|              9.37|         5.67|           0|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|   